In [ ]:
import torch
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, TrainingArguments, Trainer

In [ ]:
SPLITS = [
    "train-original",
    "train-augmented-wordnet",
    "train-augmented-lesk",
    "train-augmented-bert",
    "train-augmented-hybrid"
]

In [ ]:
# DIRNAME = "roberta"
# MODELNAME = "FacebookAI/roberta-base"
# DIRNAME = "bert"
# MODELNAME = "google-bert/bert-base-uncased"
# DIRNAME = "gpt2"
# MODELNAME = "openai-community/gpt2"
DIRNAME = "bart"
MODELNAME = "facebook/bart-large"

In [ ]:
def eval_model(model, tokenizer, ds):
    tp = 0
    tn = 0
    fp = 0
    fn = 0

    correct = [0,0]
    guesses = [0,0]

    for row in ds:
        prompt = row['text']
        label = row['label']

        with torch.no_grad():
            tokens = tokenizer(prompt, return_tensors='pt').to('cuda')
            tokenized_output = model(**tokens)
        
        guess = tokenized_output['logits'].argmax()
        correct[label] += 1
        guesses[guess] += 1
        if guess:
            if label:
                tp += 1
            else:
                fp += 1
        else:
            if label:
                fn += 1
            else:
                tn += 1
    # 
    # Print Results
    # 
    print(f"{'Results':^20}")
    print(f"Accuracy:   {(tp+tn)/len(ds):<10.2%}")
    print(f"Precision:  {tp/(tp+fp):<10.2%}")
    print(f"Recall:     {tp/(tp+fn):<10.2%}")
    print(f"F1 Score:   {(2*tp)/(2*tp+fp+fn):<10.2%}")
    print(f"Correct:    [{correct[0]}, {correct[1]}]")
    print(f"Guesses:    [{guesses[0]}, {guesses[1]}]")

In [ ]:
for sname in SPLITS:
    # 
    # Load Objects
    # 
    model = AutoModelForSequenceClassification.from_pretrained(MODELNAME)
    tokenizer = AutoTokenizer.from_pretrained(MODELNAME)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    if DIRNAME == 'gpt2':
        tokenizer.pad_token = tokenizer.eos_token
        model.config.pad_token_id = model.config.eos_token_id
    # 
    # Load & Prepare Data
    # 
    train_dataset = load_from_disk("dataset.hf")[sname]
    eval_dataset = load_from_disk("dataset.hf")['validation']
    tokenized_train_dataset = train_dataset.map(lambda ds: tokenizer(ds['text']), batched=True, remove_columns=["text"])
    # 
    # Train
    # 
    training_arguments = TrainingArguments(
        output_dir=f"models/{DIRNAME}" + sname[sname.find('-'):],
        num_train_epochs=3,
        save_strategy="epoch"
    )
    trainer = Trainer(
        model=model,
        args=training_arguments,
        train_dataset=tokenized_train_dataset,
        data_collator=data_collator
    )
    trainer.train()
    eval_model(model, tokenizer, eval_dataset)